# ProDy on Google Colab — A Guided Tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prody/ProDy/blob/claude/prody-colab-notebook-VFYRv/docs/notebooks/ProDy_Colab_Tutorial.ipynb)

[**ProDy**](http://www.bahargroup.org/prody/) is a free, open-source Python
package for **protein structure, dynamics, and sequence analysis**. It bundles
together elastic network models (ANM, GNM, exANM, exGNM, imANM, RTB), normal
mode analysis (NMA), principal component analysis (PCA) of conformational
ensembles, hybrid sampling methods (ClustENMD, ANMD), structure parsing
(PDB / mmCIF / EMD), VMD-style atom selections, trajectory I/O, the **Evol**
suite for sequence co-evolution, residue-level stiffness, perturbation
response scanning (PRS), essential site scanning analysis (ESSA), signature
dynamics (SignDy), interaction analysis (InSty), and water-bridge networks
(WatFinder) — all behind one consistent Python API.

This notebook is a guided, runnable tour of every major ProDy modality
listed on the [official tutorials page](http://www.bahargroup.org/prody/tutorials/),
designed to work end-to-end in a fresh Google Colab runtime. Each section
opens with a short explainer that covers:

- **What it does** — the analysis or task in plain language.
- **What it needs** — the inputs you have to supply.
- **What to watch for** — limitations, pitfalls, or scaling concerns.

If you are running this on Colab, just choose **Runtime → Run all** and read
along. If you are reading on GitHub, click the badge above to launch it.

> **Want to use your own structure?** Section 2 shows three ways to load a
> local `.pdb` / `.cif` file (Colab upload, Google Drive, or just a different
> PDB ID). Every downstream cell uses a single variable, `protein`, so
> swapping the input automatically flows through the rest of the notebook.

## Contents

**Part I — Core workflow**
1. Setup & installation
2. Bringing your own structure (PDB / mmCIF) — *optional*
3. Parsing the chosen structure
4. Atom selections
5. Interactive 3D view (py3Dmol)
6. ANM (Anisotropic Network Model)
7. GNM (Gaussian Network Model)
8. PCA on a multi-conformer ensemble
9. Comparing experiment vs theory (PCA / ANM overlap)

**Part II — Extending the ENM toolbox**
10. Mechanical Stiffness Matrix
11. Membrane ANM (`exANM` / `exGNM` / `imANM`)
12. Adaptive ANM — pathways between conformations
13. Perturbation Response Scanning (PRS)
14. Essential Site Scanning Analysis (ESSA)
15. Signature Dynamics (SignDy)

**Part III — Hybrid sampling (require OpenMM)**
16. ClustENMD — ENM + MD conformer generation
17. ANMD — ANM-driven minimised conformers

**Part IV — Cryo-EM coarse-graining**
18. CryoDy — `parseEMD` + ENM on density-derived beads

**Part V — Molecular interactions**
19. InSty — hydrogen bonds, salt bridges, hydrophobics, …
20. WatFinder — water-bridge networks

**Part VI — Trajectories**
21. DCD trajectories

**Part VII — Sequence evolution**
22. Conservation (Shannon entropy on a Pfam MSA)
23. Coevolution (mutual information, OMES, direct information)

**Part VIII — Other**
24. CoMD — note on the collective MD workflow
25. Saving and downloading results
26. Where to go next


## 1. Setup & installation

**What it does:** installs ProDy and the optional 3D-viewer dependency
`py3Dmol` into the Colab runtime.

**What it needs:** an active internet connection. Colab already ships
`numpy`, `scipy`, `matplotlib`, and `biopython`, which are ProDy's only
required runtime dependencies, so the install is fast (~30 s) and never
needs `conda`.

**What to watch for:**

- Colab runtimes are ephemeral — re-run this cell whenever the runtime is
  recycled.
- A handful of advanced workflows in this notebook need *optional* extras:
  `openmm` and `pdbfixer` for **ClustENMD** and **ANMD**, and
  `openbabel-wheel` for adding hydrogens before some **InSty** analyses.
  Those installs are deferred to their own sections so you only pay for
  what you need.


In [ ]:
!pip install -q prody py3Dmol


In [ ]:
import prody
import py3Dmol
import numpy as np
import matplotlib.pyplot as plt

print("ProDy version:", prody.__version__)
print("NumPy version:", np.__version__)


## 2. Bringing your own structure (PDB / mmCIF) — *optional*

**What it does:** lets you point the rest of the notebook at *your own*
protein instead of the default ubiquitin demo (PDB 1UBI). Every
downstream cell uses a single variable, `protein`, so swapping the input
here automatically flows through ANM, GNM, PCA, PRS, ESSA, InSty, and
the rest.

**Three ways to supply your own structure**

1. **Pick a different PDB / mmCIF ID** — set `STRUCTURE_INPUT` below to
   `'4ake'`, `'1aki'`, etc. ProDy fetches it via HTTP.
2. **Upload a local file** — uncomment the `files.upload()` block and use
   the Colab file picker. Works for `.pdb`, `.cif`, and `.pdb.gz`.
3. **Mount Google Drive** — uncomment the `drive.mount` block and point
   `STRUCTURE_INPUT` at e.g. `'/content/drive/MyDrive/my_protein.pdb'`.

**What it needs:** a 4-character PDB ID, a path to a local file, or a
URL fetched into `/content/`.

**What to watch for:**

- ProDy parses both PDB and mmCIF — the next cell auto-detects format
  from the file extension.
- Multi-chain assemblies parse fine; pass `chain='A'` to
  `parsePDB`/`parseMMCIF` if you need a single chain.
- Some downstream sections (PCA, SignDy) demonstrate ensemble analyses
  on the bundled NMR ensemble PDB 2K39. Those continue to use 2K39 even
  if you change `STRUCTURE_INPUT` here, because they need *multiple*
  conformers. Replace the `2k39` line in those sections to use your own
  multi-model PDB or trajectory.


In [ ]:
# === Choose ONE of the three options below ===

# Option 1 — a PDB / mmCIF ID (default: ubiquitin)
STRUCTURE_INPUT = '1ubi'           # try '4ake', '1ake', '2eqj', '3p3w', ...

# Option 2 — upload a local .pdb / .cif file from your computer
# from google.colab import files
# uploaded = files.upload()        # opens browser file picker
# STRUCTURE_INPUT = next(iter(uploaded))

# Option 3 — mount Google Drive and point at a file there
# from google.colab import drive
# drive.mount('/content/drive')
# STRUCTURE_INPUT = '/content/drive/MyDrive/my_protein.pdb'

print("Loading:", STRUCTURE_INPUT)


## 3. Parsing the chosen structure

**What it does:** dispatches to `parsePDB` or `parseMMCIF` based on the
file extension and returns an `AtomGroup` — ProDy's core container for
atomic data. The result is bound to the variable **`protein`** that
every later section uses.

**What it needs:** the value of `STRUCTURE_INPUT` from the previous cell.

**What to watch for:**

- For huge assemblies, pass `subset='ca'` or `chain='A'` to keep memory
  low. The default below is the full structure.
- mmCIF parsing requires a `.cif` extension on the file/path; raw IDs
  always go through `parsePDB`.


In [ ]:
import os

def load_structure(spec, **kwargs):
    """Parse a PDB ID, .pdb file, or .cif file into an AtomGroup."""
    lower = spec.lower()
    if os.path.isfile(spec) and (lower.endswith('.cif') or lower.endswith('.cif.gz')):
        return prody.parseMMCIF(spec, **kwargs)
    # parsePDB handles 4-character IDs *and* .pdb / .pdb.gz file paths
    return prody.parsePDB(spec, **kwargs)

protein = load_structure(STRUCTURE_INPUT)
if protein is None:
    raise RuntimeError(
        f"Could not parse {STRUCTURE_INPUT!r}. Check the ID or file path."
    )

print(protein)
print("Atoms:    ", protein.numAtoms())
print("Residues: ", protein.numResidues())
print("Chains:   ", protein.numChains())
print("Title:    ", protein.getTitle())


## 4. Atom selections

**What it does:** lets you carve out subsets of atoms with a VMD-like
selection grammar (`calpha`, `protein`, `resnum 1to10`, `within 5 of
chain B`, etc.). Selections return a lightweight `Selection` view, not a
copy.

**What it needs:** an `AtomGroup` and a selection string.

**What to watch for:**

- Empty selections return `None`, not an empty container — always check.
- The full grammar is documented at `prody.SELECT` and in the
  [selections reference](http://www.bahargroup.org/prody/manual/reference/atomic/select.html).


In [ ]:
calphas = protein.select('calpha')
heavy   = protein.select('protein and not hydrogen')

print("Cα atoms:           ", calphas.numAtoms() if calphas else 0)
print("Heavy protein atoms:", heavy.numAtoms() if heavy else 0)

# A residue-range example tied to the chosen structure
n_res = protein.select('protein and name CA').numResidues()
mid_lo, mid_hi = max(1, n_res // 4), min(n_res, 3 * n_res // 4)
core = protein.select(f'name CA and resnum {mid_lo}to{mid_hi}')
print(f"Core Cα subset (resnum {mid_lo}–{mid_hi}):",
      core.numAtoms() if core else 0)


## 5. Interactive 3D visualisation with py3Dmol

**What it does:** renders the protein as an interactive cartoon you can
rotate, zoom, and re-style — directly in the notebook output cell.

**What it needs:** `py3Dmol` (installed above) and an `AtomGroup`.

**What to watch for:**

- The viewer is a live JavaScript widget. If you save the notebook and
  reopen it without re-running the cell, the viewer will be blank.
- Static exports (PDF / nbviewer) likewise show nothing — re-run the cell
  in a live kernel.


In [ ]:
prody.view3D(protein, width=500, height=400)


## 6. Anisotropic Network Model (ANM)

**What it does:** treats every Cα as a bead connected to its neighbours by
springs and diagonalises the resulting Hessian. The lowest-frequency
non-zero modes describe the **collective, functionally relevant motions** a
protein is most likely to undergo — without any MD simulation.

**What it needs:** a Cα `AtomGroup` (or any coarse-grained selection).
Optional: a cutoff distance (default 15 Å) and spring constant (`gamma`).

**What to watch for:**

- ANM predicts **directions and relative magnitudes**, not absolute
  amplitudes. Compare modes across structures, do not over-interpret raw
  numbers.
- Resolution is residue-level. For atomic detail you need MD or all-atom
  NMA.
- The first 6 eigenvalues are zero (rigid-body translations + rotations)
  and are skipped by default.


In [ ]:
anm = prody.ANM(protein.getTitle() or 'protein')
anm.buildHessian(calphas, cutoff=15.0)
anm.calcModes(n_modes=20)

print(anm)
print("First non-zero mode eigenvalue:", anm[0].getEigval())
print("Collectivity of mode 1:", prody.calcCollectivity(anm[0]))


### Square fluctuations and cross-correlations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

plt.sca(axes[0])
prody.showSqFlucts(anm[:5])
axes[0].set_title("ANM square fluctuations (first 5 modes)")

plt.sca(axes[1])
prody.showCrossCorr(anm)
axes[1].set_title("ANM cross-correlation map")

plt.tight_layout()
plt.show()


### Animating the slowest mode in 3D

`view3D` accepts a `mode=` argument and will draw arrows or animate the
trajectory along that mode.


In [ ]:
prody.view3D(calphas, mode=anm[0], scale=80, width=500, height=400)


## 7. Gaussian Network Model (GNM)

**What it does:** the isotropic, scalar cousin of ANM. Each residue gets a
single mobility value per mode — cheaper to compute and often enough for
identifying flexible regions and hinge sites.

**What it needs:** the same Cα selection. Default cutoff is 10 Å.

**What to watch for:**

- GNM gives **magnitudes only** — no directions. Use ANM if you want to
  animate motion in 3D.
- Mode 1 (the slowest non-zero mode) typically separates the protein into
  two anti-correlated dynamic domains; sign flips between residues mark
  hinge points.


In [ ]:
gnm = prody.GNM(protein.getTitle() or 'protein')
gnm.buildKirchhoff(calphas, cutoff=10.0)
gnm.calcModes(n_modes=20)
print(gnm)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

plt.sca(axes[0])
prody.showMode(gnm[0])
axes[0].set_title("GNM mode 1 (slowest) — sign flips mark hinges")

plt.sca(axes[1])
prody.showContactMap(gnm)
axes[1].set_title("Cα contact map (Kirchhoff)")

plt.tight_layout()
plt.show()


## 8. Principal Component Analysis on a multi-conformer ensemble

**What it does:** extracts the dominant **observed** motions from a set of
related conformations — multiple NMR models, an MD trajectory, or
homologous X-ray structures.

**What it needs:** an `Ensemble` (or `PDBEnsemble`) of structurally
aligned conformers with the same number of atoms.

**What to watch for:**

- Conformations must be aligned first (`ensemble.superpose()`); ProDy can
  do this for you.
- PCA quality scales with ensemble size and diversity. A handful of nearly
  identical structures will give noisy modes.

**Bring your own ensemble:** this section uses **PDB 2K39**, an NMR
ensemble of ubiquitin with 116 models, because it ships as a single
multi-model PDB. To run PCA on *your* data, replace the `parsePDB('2k39',
…)` line below with one of:

- `prody.parsePDB('your_multimodel.pdb', subset='ca')` — any multi-model
  PDB you uploaded in Section 2.
- a list of homologs aligned with `prody.buildPDBEnsemble([ag1, ag2, …])`.
- coordinates from a DCD trajectory (see Section 21).


In [ ]:
ensemble_source = prody.parsePDB('2k39', subset='ca')
print("Conformations available:", ensemble_source.numCoordsets())

ensemble = prody.PDBEnsemble('PCA ensemble')
ensemble.setCoords(ensemble_source.getCoords())
ensemble.addCoordset(ensemble_source.getCoordsets())
ensemble.setAtoms(ensemble_source)
ensemble.superpose()

pca = prody.PCA('PCA on ensemble')
pca.buildCovariance(ensemble)
pca.calcModes(n_modes=20)
print(pca)
print("Variance fraction (mode 1):", prody.calcFractVariance(pca[0]))


In [ ]:
plt.figure(figsize=(8, 4))
prody.showFractVars(pca[:10])
prody.showCumulFractVars(pca[:10])
plt.title("Variance explained by PCA modes")
plt.show()


## 9. Comparing experimental (PCA) vs theoretical (ANM) motion

**What it does:** computes the overlap between PCA modes (what the
ensemble actually does) and ANM modes (what the elastic network predicts).
A diagonal-heavy overlap table is the classic sanity check that ENM-based
dynamics agree with experiment.

**What it needs:** two `ModeSet` objects of compatible dimension.

**What to watch for:** the ANM and PCA must be calculated on the **same
selection** (here, the Cα atoms used by the PCA above).


In [ ]:
anm_ens = prody.ANM('ANM on ensemble reference')
anm_ens.buildHessian(ensemble_source)
anm_ens.calcModes(n_modes=20)

prody.printOverlapTable(pca[:5], anm_ens[:5])


## 10. Mechanical Stiffness Matrix

**What it does:** turns ANM modes into a **per-residue effective spring
constant** matrix. `calcMechStiff(modes, atoms)` returns an N×N matrix
whose (i, j) entry is the stiffness felt between residue *i* and residue
*j*. Stiff residues are mechanically rigid hubs; soft residues are
deformation hot-spots.

**What it needs:** an `ANM` model with calculated modes and the matching
Cα selection.

**What to watch for:**

- Output matrix is N×N for N Cα atoms (memory ≈ 8·N² bytes — fine up to
  ~10 000 residues).
- Backed by a compiled C extension (`smtools`); installs cleanly via pip.
- A useful summary plot is `showMechStiff` for a heatmap, plus
  `calcStiffnessRange` for per-residue stats.


In [ ]:
sm = prody.calcMechStiff(anm, calphas)
print("Stiffness matrix shape:", sm.shape)
print("Stiffness range: {:.2f} — {:.2f}".format(sm[sm > 0].min(), sm.max()))

plt.figure(figsize=(6, 5))
plt.imshow(sm, cmap='viridis', origin='lower')
plt.colorbar(label='Effective spring constant')
plt.title(f"Mechanical Stiffness Matrix — {protein.getTitle()}")
plt.xlabel("Residue index")
plt.ylabel("Residue index")
plt.tight_layout()
plt.show()


## 11. Membrane ANM — `exANM`, `exGNM`, `imANM`

**What it does:** elastic network models for membrane-embedded proteins.

- `exANM` / `exGNM` — *explicit* membrane: ProDy generates a lattice of
  pseudo-atoms representing a lipid bilayer and includes them in the
  Hessian/Kirchhoff so that membrane-confined motions are correctly
  damped.
- `imANM` — *implicit* membrane: uses the rotation–translation block (RTB)
  approach to constrain transverse motions without explicit lipid
  particles.

**What it needs:**

- A protein already aligned so the membrane normal is the **z-axis** and
  the bilayer is centred at z = 0 (use the [OPM](https://opm.phar.umich.edu)
  or [PPM](https://opm.phar.umich.edu/ppm_server) servers for this).
- Lipid bilayer thickness (defaults to z ∈ [-13, 13] Å).

**What to watch for:**

- ProDy does **not** orient your structure for you — pre-align with OPM.
- exANM/exGNM scale with the number of synthetic lipid beads; for very
  large transmembrane assemblies, prefer `imANM`.
- This cell shows the **API** only because the default `protein` is not
  membrane-aligned; swap in an OPM-aligned structure to actually run.


In [ ]:
# API demonstration — replace the path with an OPM-aligned membrane
# protein to actually run buildMembrane / buildHessian.
#
# membrane_protein = prody.parsePDB('opm_aligned.pdb', subset='ca')
# exa = prody.exANM('membrane test')
# exa.buildMembrane(membrane_protein, cutoff=15., gamma=1.,
#                   membrane_high=13., membrane_low=-13.)
# exa.calcModes(n_modes=20)
# print(exa)

print("ProDy membrane-ENM classes available:")
for name in ['exANM', 'exGNM', 'imANM']:
    cls = getattr(prody, name)
    first_line = (cls.__doc__ or '').splitlines()[0] if cls.__doc__ else cls
    print(f"  prody.{name}  →  {first_line}")


## 12. Adaptive ANM — pathways between two conformations

**What it does:** generates a smooth pathway connecting two end-state
conformations (e.g. *open* ↔ *closed*) by iteratively deforming along the
ANM modes that overlap most with the structural difference.

`prody.calcAdaptiveANM(a, b, n_steps, mode=AANM_BOTHWAYS)` returns an
ensemble of intermediates plus per-step ANM models.

**What it needs:** two `Atomic` objects (or coordinate arrays) with the
**same atom count** (typically the matching Cα selection of two states).

**What to watch for:**

- The two structures must be matched residue-by-residue. Use
  `prody.matchAlign` or `matchChains` if sequences differ.
- Convergence is measured by RMSD reduction; tune `Fmin` (cumulative
  overlap threshold) and `f` (step scaling) for slow convergence.
- Modes: `AANM_ONEWAY` (a→b only), `AANM_ALTERNATING` (zig-zag), or
  `AANM_BOTHWAYS` (a→b plus b→a, joined at the middle — recommended).

For a quick demo we use two distinct conformers from the 2K39 NMR
ensemble (guaranteed atom-count match). To use *your own* end states,
replace `state_a` / `state_b` with two parsed PDBs of matching length.


In [ ]:
# Two distinct conformers from the 2K39 NMR ensemble
state_a = ensemble_source.copy()
state_a.setCoords(ensemble_source.getCoordsets()[0])
state_b = ensemble_source.copy()
last_idx = ensemble_source.numCoordsets() - 1
state_b.setCoords(ensemble_source.getCoordsets()[last_idx])

initial_rmsd = prody.calcRMSD(state_a, state_b)
print(f"Initial RMSD: {initial_rmsd:.2f} Å")

aanm_ensemble = prody.calcAdaptiveANM(
    state_a, state_b, n_steps=5, mode=prody.AANM_BOTHWAYS
)
print(aanm_ensemble)


## 13. Perturbation Response Scanning (PRS)

**What it does:** probes how the protein **propagates** a force applied at
each residue. The output `prs_matrix[i, j]` is the response at residue *j*
to a perturbation at residue *i*. Two derived 1-D profiles fall out of it:

- **Effectiveness** (rows): residues that, when perturbed, move the rest
  of the protein the most. These are typical *driver* / *sensor* sites.
- **Sensitivity** (columns): residues most easily moved by perturbations
  elsewhere. These are typical *responsive* / *allosteric* sites.

**What it needs:** an NMA model (`ANM`, `GNM`, `PCA`) with calculated
modes, plus the matching `Atomic` selection.

**What to watch for:**

- PRS quality is bounded by the input modes — use enough modes to capture
  most of the variance (≥ 20 is typical).
- `turbo=True` (default) is fast but uses more RAM; set `turbo=False` and
  `repeats=N` for the classical random-perturbation algorithm.


In [ ]:
prs_matrix, eff, sens = prody.calcPerturbResponse(anm, atoms=calphas)
print("PRS matrix shape:", prs_matrix.shape)
print("Effectiveness:    ", eff.shape)
print("Sensitivity:      ", sens.shape)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

im = axes[0].imshow(prs_matrix, cmap='hot', origin='lower')
axes[0].set_title("PRS matrix — response to perturbation")
axes[0].set_xlabel("Sensor residue")
axes[0].set_ylabel("Perturbed residue")
plt.colorbar(im, ax=axes[0])

axes[1].plot(eff,  label='Effectiveness')
axes[1].plot(sens, label='Sensitivity')
axes[1].set_xlabel("Residue")
axes[1].set_ylabel("PRS profile")
axes[1].legend()
axes[1].set_title("Driver vs responsive sites")

plt.tight_layout()
plt.show()


## 14. Essential Site Scanning Analysis (ESSA)

**What it does:** for each residue, ProDy stiffens its surroundings (adds
springs to its neighbours) and re-computes the global modes. The
**z-score** of the resulting eigenvalue shift quantifies how *essential*
that residue is for global dynamics. High z-scores flag residues whose
local environment dictates collective motion — often catalytic or
allosteric hot-spots.

**What it needs:** an `AtomGroup` for the protein (and optionally a
ligand selection string for ligand-aware scoring).

**What to watch for:**

- Default analysis uses GNM with 10 modes; use `enm='anm'` for
  directional information.
- Ligand-binding pocket ranking via `essa.scanPockets()` requires the
  external **fpocket** binary; the residue scan does not.


In [ ]:
essa = prody.ESSA()
essa.setSystem(protein)
essa.scanResidues(n_modes=10, enm='gnm')
zscores = essa.getESSAZscores()

plt.figure(figsize=(10, 3))
plt.plot(zscores)
plt.axhline(2, color='red', linestyle='--', label='z = 2 cutoff')
plt.xlabel("Residue index")
plt.ylabel("ESSA z-score")
plt.title(f"Essential residues (ESSA) — {protein.getTitle()}")
plt.legend()
plt.tight_layout()
plt.show()

top = np.argsort(zscores)[-5:][::-1]
print("Top 5 essential residues (0-indexed):", top.tolist())


## 15. Signature Dynamics (SignDy)

**What it does:** runs an ENM on **every** member of an ensemble of
homologs (or NMR conformers, or MD frames) and aligns the resulting modes
into a `ModeEnsemble`. From that you get *signature* fluctuations,
collectivity, and overlap statistics that summarise how the family
behaves dynamically as a whole, not just one structure.

**What it needs:** a `PDBEnsemble` (note: PDBEnsemble, not plain
Ensemble — it carries per-conformer labels needed by SignDy).

**What to watch for:**

- Computational cost: one ENM per conformer + a mode-matching pass.
  Manageable for tens of structures, expensive for thousands.
- For a homolog family, build the `PDBEnsemble` with
  `prody.buildPDBEnsemble` from a list of structures returned by
  `blastPDB` or `searchDali`.


In [ ]:
# Reuse the multi-conformer PDBEnsemble built in section 8
mode_ens = prody.calcEnsembleENMs(ensemble, model='ANM', n_modes=10)
print(mode_ens)
print("ModeEnsemble has", len(mode_ens), "ENMs")

sig_flucts = prody.calcSignatureSqFlucts(mode_ens)
print("Signature SqFlucts shape:", np.asarray(sig_flucts).shape)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

plt.sca(axes[0])
prody.showSignatureSqFlucts(mode_ens)
axes[0].set_title("Signature square fluctuations (mean ± std)")

plt.sca(axes[1])
prody.showSignatureMode(mode_ens)
axes[1].set_title("Signature mode 1 profile")

plt.tight_layout()
plt.show()


## 16. ClustENMD — ENM-driven conformer generation + MD

**What it does:** generates a diverse, atomistic conformational ensemble
by alternating **ENM mode sampling** (random kicks along the slowest
modes) with **clustering** (to remove redundancy) and **MD relaxation**
in OpenMM (to produce physically reasonable structures). Several
*generations* of this cycle widen the explored basin without the cost of
a long MD simulation.

**What it needs:**

- A clean atomic structure (no missing residues / atoms — `PDBFixer` is
  used internally to repair).
- `openmm` and `pdbfixer` Python packages.

**What to watch for:**

- **Heavy.** A full run with ~20 conformers per generation × 2 generations
  on ubiquitin takes minutes on a CPU and seconds on a Colab GPU runtime
  (`Runtime → Change runtime type → GPU`).
- ClustENMD writes intermediate PDB files and a `.dcd` trajectory in the
  working directory.
- Linux-only (tested); Windows may need WSL.


In [ ]:
# Optional: install OpenMM + PDBFixer (~1 min on Colab CPU). Uncomment
# the next line and the run cell below to actually execute.
#
# !pip install -q openmm pdbfixer


In [ ]:
# from prody import ClustENM
# ce = ClustENM()
# ce.setAtoms(protein)
# ce.run(n_modes=3, n_confs=20, n_gens=2, rmsd=1.5)
# print("Generated", ce.numConfs(), "conformers across", ce.numGenerations(),
#       "generations")
# # Save as a multi-model PDB / DCD
# prody.writePDB('clustenm_protein.pdb', ce)

print("ClustENMD demo is commented out by default — uncomment after"
      " installing openmm + pdbfixer.")
print("ClustENM class is available at:", prody.ClustENM)


## 17. ANMD — ANM-driven minimised conformers

**What it does:** for each of the slowest ANM modes, ProDy displaces the
protein along that mode in increments, *energy-minimises* each step in
OpenMM with the AMBER99SBildn force field + OBC implicit solvent, and
returns the resulting per-mode trajectories. Useful for visualising
realistic-looking motion without running MD.

**What it needs:** a complete atomic structure (loops intact) and OpenMM.

**What to watch for:**

- Each conformer is minimised independently — runtime scales linearly
  with `num_modes × num_steps × 2` (both directions).
- Output is a `list[Ensemble]`, one ensemble per mode.


In [ ]:
# Optional: install OpenMM (~1 min)
# !pip install -q openmm


In [ ]:
# from prody import runANMD
# ensembles = runANMD(protein, num_modes=2, max_rmsd=2.0, num_steps=3)
# for i, ens in enumerate(ensembles):
#     print(f"Mode {i+1}: {ens.numConfs()} conformers,"
#           f" RMSD range {ens.getRMSDs().min():.2f}–{ens.getRMSDs().max():.2f} Å")

print("runANMD demo is commented out by default — uncomment after"
      " installing openmm.")
print("Function available at:", prody.runANMD)


## 18. CryoDy — coarse-graining cryo-EM density maps

**What it does:** ProDy can read **EMD/MRC** density maps and
*automatically* place a user-specified number of pseudo-atom beads
inside the density using the **Topology-Representing Network (TRN)**
algorithm. The resulting `AtomGroup` plugs straight into ANM/GNM, so you
can do dynamics on cryo-EM densities for which no atomic model is
available.

**What it needs:** a path to a `.map` / `.mrc` / `.emd` file or an
EMD-DB ID (e.g. `'1234'` for `EMD-1234`).

**What to watch for:**

- Network access for ID-based fetches.
- Choose `n_nodes` carefully: too few loses detail, too many blows up
  ANM cost.
- The output beads are **not** real atoms; they're placeholders for
  density. Treat distance-based selections accordingly.


In [ ]:
# API demo — uncomment and supply a real EMD ID or local map to run.
#
# beads = prody.parseEMD('1234', n_nodes=200)        # auto-coarse-grained
# anm_cryo = prody.ANM('cryoEM beads')
# anm_cryo.buildHessian(beads, cutoff=15.0)
# anm_cryo.calcModes(n_modes=20)
# print(anm_cryo)

print("parseEMD signature:", prody.parseEMD)


## 19. InSty — Interactions and Stability

**What it does:** detects and characterises non-covalent interactions
inside (or between) protein structures: hydrogen bonds, salt bridges,
repulsive ionic contacts, π–π stacking, π–cation, hydrophobic contacts,
disulfide bonds, and metal coordination. Each call returns a list of
interaction tuples with geometric metrics (distance, angle).

**What it needs:**

- An `AtomGroup`. **Hydrogen bonds** require explicit hydrogens —
  `addMissingAtoms` (which uses OpenBabel) can add them, or use an NMR
  structure that already has them.
- The other interaction types (salt bridges, hydrophobics, disulfides,
  π–π, π–cation) work on heavy-atom-only structures.

**What to watch for:**

- ProDy ships precompiled `hpb.so` for hydrophobic-interaction analysis on
  multiple Python versions but not all platforms; a fall-back
  geometric calculation is used otherwise.
- Trajectory variants (`calcHydrogenBondsTrajectory`, …) iterate frame by
  frame — expect long runtimes on big trajectories.


In [ ]:
# These calls work on heavy-atom-only X-ray structures (no H needed)
salt_bridges = prody.calcSaltBridges(protein)
disulfides   = prody.calcDisulfideBonds(protein)
hydrophobic  = prody.calcHydrophobic(protein)
pistack      = prody.calcPiStacking(protein)

print(f"Salt bridges:           {len(salt_bridges) if salt_bridges else 0}")
print(f"Disulfide bonds:        {len(disulfides) if disulfides else 0}")
print(f"Hydrophobic contacts:   {len(hydrophobic) if hydrophobic else 0}")
print(f"π-π stacking:           {len(pistack) if pistack else 0}")


In [ ]:
# Hydrogen bonds — needs hydrogens. Uncomment the openbabel install
# and addMissingAtoms call to enable on a heavy-atom-only structure.
#
# !pip install -q openbabel-wheel
# protein_h = prody.addMissingAtoms(protein, ph=7.0)
# hbonds = prody.calcHydrogenBonds(protein_h, distDA=3.5, angleDHA=40)
# print(f"Hydrogen bonds: {len(hbonds)}")


## 20. WatFinder — water-bridge networks

**What it does:** finds chains of water molecules that bridge protein
residues via hydrogen bonds. Either a single short chain
(`method='chain'`) or a graph-clustered network (`method='cluster'`)
can be requested. Useful for characterising solvent-mediated allostery,
substrate channels, and tightly-bound structural waters.

**What it needs:** a structure that **contains explicit water**
(residue name `HOH` / `WAT`). Most high-resolution X-ray PDBs do; NMR
and EM structures usually don't.

**What to watch for:**

- Computational cost grows quickly with the number of waters; consider
  `protein.select('protein or (water within 8 of protein)')` to keep only
  near-protein waters.
- `output='info'` returns a tidy human-readable list; `output='atomic'`
  returns the raw atom indices for further processing.


In [ ]:
waters = protein.select('water')
print(f"Water molecules: {waters.numAtoms() if waters else 0}")

if waters and waters.numAtoms() > 0:
    bridges = prody.calcWaterBridges(protein, method='chain', distDA=3.5)
    print(f"Found {len(bridges)} water bridges")
    print("First 3 bridges (raw atomic output):")
    for b in bridges[:3]:
        print(" ", b)
else:
    print("No waters in this structure — try a high-resolution X-ray PDB"
          " (e.g. '1ubi', '1aki') to see WatFinder in action.")


## 21. Working with DCD trajectories

**What it does:** ProDy reads and writes CHARMM/NAMD **DCD** trajectory
files and feeds them straight into PCA/EDA workflows.

**What it needs:** a `.dcd` file plus the matching topology (PDB).

**What to watch for:**

- Only DCD is native. For `.xtc` / `.trr` (GROMACS) or `.nc` (AMBER), use
  [MDAnalysis](https://www.mdanalysis.org/) or
  [mdtraj](https://www.mdtraj.org/) and pass the coordinates into a
  `prody.Ensemble`.
- Big trajectories should be processed frame-by-frame
  (`for frame in trajectory:`) instead of loading every coordinate set
  into RAM.

To use your own trajectory, upload `topology.pdb` and `trajectory.dcd`
via Section 2 (or mount Drive) and uncomment the cell below.


In [ ]:
# Example only — supply your own topology + DCD to run.
#
# topology   = prody.parsePDB('topology.pdb')
# trajectory = prody.Trajectory('trajectory.dcd')
# trajectory.link(topology)
# trajectory.setAtoms(topology.calpha)
#
# eda = prody.EDA('MD essential dynamics')
# eda.buildCovariance(trajectory)
# eda.calcModes(n_modes=20)
# print(eda)


## 22. Sequence conservation with Evol

**What it does:** ProDy's **Evol** subpackage downloads multiple sequence
alignments from Pfam and computes per-residue conservation (Shannon
entropy). Conserved residues often coincide with mechanically important
sites identified by ANM/GNM.

**What it needs:** a Pfam family accession (e.g. `PF00240` for
ubiquitin) **or** a UniProt ID to search Pfam with.

To use *your own* MSA, replace the `fetchPfamMSA` line with
`msa = prody.parseMSA('your_alignment.fasta')` (FASTA, Stockholm, or
SELEX).

**What to watch for:**

- Pfam and InterPro APIs change occasionally; if the fetch fails the rest
  of the notebook is unaffected — skip this section.
- MSAs for popular families can be large (tens of MB).


In [ ]:
msa = None
try:
    msa_path = prody.fetchPfamMSA('PF00240', alignment='seed')
    msa = prody.parseMSA(msa_path)
    msa = prody.refineMSA(msa, rowocc=0.8, seqid=0.98)
    entropy = prody.calcShannonEntropy(msa)

    plt.figure(figsize=(10, 3))
    plt.plot(entropy)
    plt.xlabel("Alignment column")
    plt.ylabel("Shannon entropy")
    plt.title("Conservation across the ubiquitin Pfam family (PF00240)")
    plt.tight_layout()
    plt.show()
except Exception as exc:
    print("Pfam fetch unavailable in this runtime:", exc)


## 23. Sequence coevolution — MI, OMES, direct information

**What it does:** ProDy computes per-pair coevolution scores from an
MSA. The most popular methods are:

- **`buildMutinfoMatrix`** — Shannon mutual information (MI). Simple,
  noisy.
- **`applyMutinfoCorr(mi, corr='apc')`** — average-product correction
  on MI; reduces phylogenetic / entropy bias.
- **`buildOMESMatrix`** — observed-minus-expected statistic.
- **`buildDirectInfoMatrix`** — direct information (mean-field DCA).
  Best signal for inferring physical contacts but needs ≥ 250 sequences.

High coevolution scores often correspond to spatially close residues or
to functionally coupled allosteric pairs.

**What it needs:** an MSA loaded by `parseMSA` (FASTA, Stockholm, or
SELEX).

**What to watch for:**

- Apply APC correction (`applyMutinfoCorr(..., corr='apc')`) for cleaner
  MI signals.
- DI is sensitive to MSA size — small / redundant alignments produce
  numerically unstable matrices.


In [ ]:
# Reuse the Pfam MSA from section 22 (if available); otherwise download
# again with seed alignment.
if msa is None:
    msa = prody.parseMSA(prody.fetchPfamMSA('PF00240', alignment='seed'))
    msa = prody.refineMSA(msa, rowocc=0.8, seqid=0.98)

mi      = prody.buildMutinfoMatrix(msa)
mi_apc  = prody.applyMutinfoCorr(mi, corr='apc')
omes    = prody.buildOMESMatrix(msa)

print("MSA shape:", msa.numSequences(), "x", msa.numResidues())
print("MI:   ", mi.shape, "max =", mi.max())
print("MI(APC):", mi_apc.shape, "max =", mi_apc.max())
print("OMES: ", omes.shape, "max =", omes.max())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
im0 = axes[0].imshow(mi_apc, cmap='hot', origin='lower')
axes[0].set_title("Mutual information (APC corrected)")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(omes, cmap='hot', origin='lower')
axes[1].set_title("OMES coevolution")
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()


## 24. CoMD — Collective Molecular Dynamics (note)

**What it is:** **Collective Molecular Dynamics (CoMD)** combines ANM mode
sampling with targeted/steered MD to drive a protein between two
conformations. It is described in:

> Gur M, Madura JD, Bahar I. *Global transitions of proteins explored by a
> multiscale hybrid methodology: application to adenylate kinase.*
> Biophysical Journal **2013**, 105(7):1643-1652.

**Why it's only a note here:** CoMD is **not** packaged as a Python module
in pip-installable ProDy. The reference implementation is an external
NAMD-driven pipeline (Tcl + Python glue scripts) maintained at
<http://www.bahargroup.org/comd/>. To run CoMD you need NAMD, VMD/ProDy,
and the CoMD scripts — outside the scope of a Colab tutorial.

**ProDy alternatives in this notebook** that fill a similar niche without
NAMD:

- `calcAdaptiveANM` (Section 12) — pathway between two conformers using
  ANM only.
- `ClustENMD` (Section 16) — ENM + OpenMM hybrid sampling.
- `runANMD` (Section 17) — ANM-driven minimised conformers.


## 25. Saving and downloading results

**What it does:** ProDy persists models in a few formats:

- `saveModel()` — pickled `.npz` of an ANM/GNM/PCA result you can reload
  later with `loadModel()`.
- `writeNMD()` — text format read by **NMWiz**, the VMD plugin for
  visualising normal modes.
- `writePDB()` — standard PDB output.
- `writeMMCIF()` — write back to mmCIF.

**What to watch for:** Colab's filesystem is wiped when the runtime is
recycled. Use `google.colab.files.download(...)` to pull files to your
local machine, or mount Google Drive.


In [ ]:
prody.saveModel(anm, 'protein_anm')
prody.writeNMD('protein_anm.nmd', anm[:5], calphas)
prody.writePDB('protein_ca.pdb', calphas)

import os
for f in ['protein_anm.anm.npz', 'protein_anm.nmd', 'protein_ca.pdb']:
    if os.path.exists(f):
        print(f"{f}\t{os.path.getsize(f):,} bytes")


In [ ]:
# Uncomment to download the saved files locally when running on Colab:
#
# from google.colab import files
# files.download('protein_anm.nmd')
# files.download('protein_ca.pdb')


## 26. Where to go next

This notebook covers every modality listed on the [official tutorials
page](http://www.bahargroup.org/prody/tutorials/) (except the
Scipion-EM-ProDy GUI integration, which lives outside Colab).

**Next steps you might explore on your own**

- **NMWiz** — VMD plugin for interactive 3D normal-mode visualisation;
  load any `.nmd` file written by `writeNMD` (Section 25).
  <http://www.bahargroup.org/prody/nmwiz>
- **DruGUI** — druggability simulations and probe analysis from MD,
  shipped inside ProDy as `prody.drugui`.
- **ChromDy / Hi-C** — chromatin dynamics from contact maps,
  `prody.chromatin`.
- **Spectrus** — dynamical domain decomposition,
  `prody.domain_decomposition`.
- **Database interfaces** — `prody.blastPDB`, `prody.searchDali`,
  `prody.searchPfam`, `prody.searchUniprotID`, `prody.parseCATH`.
- **Scipion-EM-ProDy** — cryo-EM-aware ProDy plugin for the Scipion
  workflow engine: <https://scipion-em.github.io/docs/>.

**Resources**

- Full tutorial index — <http://www.bahargroup.org/prody/tutorials/>
- API reference — <http://www.bahargroup.org/prody/manual>
- Source / issues — <https://github.com/prody/ProDy>

If you use ProDy in published work, please cite the *Bioinformatics*
papers listed in the [README](https://github.com/prody/ProDy#readme).
